# Data Preprocessing & Feature Engineering: Credit Card Fraud Detection

**Objective:**
To transform raw data into an optimal mathematical representation for *machine learning* algorithms.

Based on the findings of *Exploratory Data Analysis* (EDA), we face several structural challenges in this dataset:
1. **Multicollinearity & Non-Standard Data Types:** Geospatial coordinates and timestamps cannot be directly processed by *tree-based models*.
2. **Data Leakage & Noise:** The presence of the “fraud_” prefix in *merchant* names and *Personally Identifiable Information* (PII) attributes.
3. **Curse of Dimensionality:** High cardinality in categorical features (`merchant`, `job`).
4. **Extreme Skewness & Outliers:** The distribution of transaction values (`amt`) has a very long *tail*.
5. **Severe Class Imbalance:** The minority class (Fraud) accounts for only 0.58%.

This workbook will address these issues sequentially.

In [2]:
import pandas as pd
import numpy as np
import math
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
import category_encoders as ce

warnings.filterwarnings('ignore')

train_path = '../dataset/01_raw/fraudTrain.csv'
df = pd.read_csv(train_path)

if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print(f"Initial Dataset Shape: {df.shape}")

Initial Dataset Shape: (1296675, 22)


## Phase 1: Feature Extraction (Spatial & Temporal)

**Problem:**
The predictive model lacks the spatial capability to calculate the distance between two pairs of coordinates (`lat`, `long` and `merch_lat`, `merch_long`), and cannot parse temporal patterns from *datetime string* formats.

**Solution:**
Extract deterministic features based on domain *knowledge*:
- **Haversine Distance:** Convert the 4-dimensional coordinates into a 1-dimensional linear distance (kilometers).
- **Age:** Transform a static proxy (`dob`) into a dynamic metric (Age).
- **Temporal Features:** Break down `trans_date_trans_time` into hour and day variables to capture activity cycles.

In [3]:
# 1.1 Temporal Extraction
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['dob'] = pd.to_datetime(df['dob'])

df['transaction_hour'] = df['trans_date_trans_time'].dt.hour
df['transaction_day'] = df['trans_date_trans_time'].dt.dayofweek # 0 = Monday, 6 = Sunday
df['age'] = (df['trans_date_trans_time'] - df['dob']).dt.days // 365

# 1.2 Spatial Extraction (Haversine Formula)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0 # Radius bumi dalam km
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

df['distance_km'] = df.apply(
    lambda row: haversine(row['lat'], row['long'], row['merch_lat'], row['merch_long']), 
    axis=1
)

## Phase 2: Data Cleansing & Dimensionality Reduction

**Problem:**
1. *Data Leakage*: It was discovered that the `merchant` column has the prefix “fraud_,” which is an artifact from the Sparkov simulator. If left unchecked, the model will learn this artifact rather than the fundamental patterns of fraud.
2. *Noise*: The PII columns (name, card number) and unique IDs have 100% variance, which would cause severe *overfitting*.

**Solution:**
Perform text cleansing (Regex) on the `merchant` attribute and reduce dimensionality by dropping variables that have no statistically significant predictive value.

In [5]:
# 2.1 Membersihkan Data Leakage
df['merchant'] = df['merchant'].str.replace('fraud_', '')

# 2.2 Menghapus Fitur Redundan dan PII
cols_to_drop = [
    'cc_num', 'first', 'last', 'street', 'city', 'state', 'zip', 
    'lat', 'long', 'merch_lat', 'merch_long', 'dob', 'trans_num', 
    'unix_time', 'trans_date_trans_time'
]

df_clean = df.drop(columns=cols_to_drop)
print(f"Shape setelah reduksi dimensi: {df_clean.shape}")

Shape setelah reduksi dimensi: (1296675, 11)


## Phase 3: Categorical Encoding

**Problem:** 
Dataset menyisakan fitur kategorikal dengan kardinalitas beragam (Kategori: 14 unik, Merchant: 693 unik, Job: 494 unik). Penggunaan *One-Hot Encoding* (OHE) pada fitur kardinalitas tinggi akan menciptakan matriks *sparse* raksasa (*Curse of Dimensionality*).

**Solution:** 
- **Low Cardinality** (`gender`, `category`): *One-Hot Encoding*.
- **High Cardinality** (`merchant`, `job`): *Target Encoding* (menghitung probabilitas historis terjadinya *fraud* pada masing-masing entitas tersebut). Diterapkan *smoothing* untuk mencegah *overfitting* pada frekuensi rendah.

In [7]:
# Pemisahan Prediktor dan Target untuk Encoding
X = df_clean.drop(columns=['is_fraud'])
y = df_clean['is_fraud']

# 3.1 One-Hot Encoding untuk Low Cardinality
X = pd.get_dummies(X, columns=['gender', 'category'], drop_first=True)

# 3.2 Target Encoding untuk High Cardinality
# Menggunakan category_encoders untuk mengelola smoothing secara matematis
target_encoder = ce.TargetEncoder(cols=['merchant', 'job'], smoothing=10)
X[['merchant', 'job']] = target_encoder.fit_transform(X[['merchant', 'job']], y)

display(X.head(3))

,merchant,amt,city_pop,job,transaction_hour,transaction_day,age,distance_km,gender_M,category_food_dining,...,category_grocery_pos,category_health_fitness,category_home,category_kids_pets,category_misc_net,category_misc_pos,category_personal_care,category_shopping_net,category_shopping_pos,category_travel
0,0.014207,4.97,3495,0.001693,0,1,30,78.597568,False,False,...,False,False,False,False,True,False,False,False,False,False
1,0.010787,107.23,149,0.002157,0,1,40,30.212176,False,False,...,True,False,False,False,False,False,False,False,False,False
2,0.002111,220.11,4154,0.015656,0,1,56,108.206083,True,False,...,False,False,False,False,False,False,False,False,False,False


## Phase 4: Feature Scaling (Handling Skewness & Outliers)

**Problem:**
Based on the descriptive statistics from EDA, `amt` (Transaction Amount) is highly *skewed*, with a range from $1 to $28,948, but the third quartile is only at $83.14. Using a conventional *StandardScaler* would be severely distorted by this extreme (maximum) value.

**Solution:**
Implement `RobustScaler`. This method uses the *Interquartile Range* (IQR) instead of the *mean* and variance, making it immune to *outliers*, which are a characteristic of high-value *fraud* transactions.

In [8]:
# 4.1 Inisialisasi Scaler
# Hanya fitur berdistribusi kontinu ekstrem yang diskalakan
cols_to_scale = ['amt', 'city_pop', 'distance_km', 'age']

robust_scaler = RobustScaler()
X[cols_to_scale] = robust_scaler.fit_transform(X[cols_to_scale])

## Phase 5: Stratified Data Splitting & Prevention of Data Leakage

**Problem:**
Only 0.58% of the data is labeled as *fraud*. A random *train-test split* risks unevenly distributing the representation of the minority class, compromising the validity of the evaluation. Additionally, *oversampling* techniques (such as SMOTE) must not be applied before the *splitting* phase, as they carry a high risk of introducing *data leakage* into the *testing set*.

**Solution:**
Perform a *Stratified Train-Test Split* to maintain the 99.42% : 0.58% ratio in both datasets. *Class imbalance handling* will be managed exclusively during the *Machine Learning* architecture development phase through *Class Weights* or in-pipeline SMOTE.

In [9]:
# 5.1 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y # Kasus dataset imbalanced
)

print(f"Training Set Shape: {X_train.shape}")
print(f"Testing Set Shape: {X_test.shape}")
print("\nVerifikasi Rasio Fraud setelah Split:")
print(f"Train Fraud Rate: {y_train.mean():.4f}")
print(f"Test Fraud Rate: {y_test.mean():.4f}")

# 5.2 Menyimpan Artifacts Preprocessed (Opsional, untuk digunakan di Model Training)
import os
os.makedirs('../dataset/02_processed', exist_ok=True)

# Menggabungkan kembali sementara untuk diekspor
train_processed = pd.concat([X_train, y_train], axis=1)
test_processed = pd.concat([X_test, y_test], axis=1)

train_processed.to_parquet('../dataset/02_processed/train_processed.parquet')
test_processed.to_parquet('../dataset/02_processed/test_processed.parquet')

# Data diekspor ke format Parquet untuk komputasi I/O.

Training Set Shape: (1037340, 22)
Testing Set Shape: (259335, 22)

Verifikasi Rasio Fraud setelah Split:
Train Fraud Rate: 0.0058
Test Fraud Rate: 0.0058
